# Day 2 — User Segmentation with K-Means

**User Engagement Intelligence**

Goal: group users into behavioral tiers using only engagement features (not churn).
We'll later use cluster membership as a feature for churn models (Day 3).

Steps:
1. Load data and select behavioral features
2. Standardize features (`StandardScaler`)
3. Choose *k* with elbow method + silhouette score
4. Fit K-Means, profile clusters, name them
5. Visualize with PCA (2D projection for plotting only)
6. Save clustered data + fitted models for Day 3

## 1. Setup & load data

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

# Make src/ importable so we can reuse CLUSTER_NAMES / paths later
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

df = pd.read_csv(PROJECT_ROOT / "data" / "users.csv")
print(f"Loaded {len(df):,} users x {df.shape[1]} columns")
df.head()

## 2. Select behavioral features

We **exclude** `churn` (that's the Day 3 label — using it here would be leakage) and `user_id` (an identifier, not a behavior).
`age` and `device_type` are demographics/context; Day 2 focuses on engagement behavior only.

In [ ]:
BEHAVIORAL_FEATURES = [
    "sessions_last_30_days",
    "avg_session_duration_minutes",
    "content_views_last_30_days",
    "likes_last_30_days",
    "shares_last_30_days",
    "days_since_last_login",
    "notifications_clicked_last_30_days",
]

X_raw = df[BEHAVIORAL_FEATURES].copy()
print("Features used for clustering:")
print(X_raw.describe().round(2).T[["mean", "std", "min", "max"]])

## 3. Standardize features

**Why scale?** K-Means minimizes Euclidean distance. Features live on different scales
(e.g. `content_views` up to ~200 vs `shares` up to ~20). Without scaling, large-range
features dominate the distance and the clusters mostly ignore smaller-scale signals.

In [ ]:
# StandardScaler: for each column, (x - mean) / std  → roughly mean 0, std 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Quick check: after scaling, means ~0 and stds ~1
scaled_check = pd.DataFrame(X_scaled, columns=BEHAVIORAL_FEATURES)
print("After StandardScaler (should be ~0 mean, ~1 std):")
print(scaled_check.agg(["mean", "std"]).round(3))

## 4. Choose *k*: elbow method + silhouette score

- **Inertia** (elbow): within-cluster sum of squared distances to the centroid.
  Lower is tighter clusters. Plot vs *k* and look for an "elbow" where adding
  more clusters stops helping much.
- **Silhouette score**: how similar a point is to its own cluster vs the nearest
  other cluster. Ranges roughly -1 to 1; **higher is better** (clearer separation).

We evaluate *k* = 2…10 with a fixed `random_state` for reproducibility.

In [ ]:
K_RANGE = range(2, 11)
inertias = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)          # lower = tighter clusters
    silhouettes.append(silhouette_score(X_scaled, labels))  # higher = better separation

metrics = pd.DataFrame(
    {"k": list(K_RANGE), "inertia": inertias, "silhouette": silhouettes}
)
metrics.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Elbow plot (inertia) ---
axes[0].plot(metrics["k"], metrics["inertia"], marker="o", color="#4c72b0")
axes[0].set_title("Elbow Method (Inertia vs k)")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster SSE)")
axes[0].set_xticks(list(K_RANGE))

# --- Silhouette plot ---
axes[1].plot(metrics["k"], metrics["silhouette"], marker="o", color="#c44e52")
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score (higher = better)")
axes[1].set_xticks(list(K_RANGE))

plt.tight_layout()
plt.show()

### Why we pick **k = 4**

- **Elbow:** Inertia drops sharply from k=2 → 3 → 4, then flattens. After k=4 the curve has diminishing returns (extra clusters buy less compactness).
- **Silhouette:** Highest at k=2 (~0.46), then declines. Pure silhouette would favor 2 clusters, but that only splits "engaged vs not" — too coarse for product tiers.
- **Interpretability trade-off:** k=4 still has a solid silhouette (~0.30) and yields four distinct behavioral tiers we can name and act on (Power / Steady / Casual / At-Risk).

**Final choice: k = 4** — balances statistical metrics with segments a retention team can use.

## 5. Fit final K-Means and assign labels

In [ ]:
FINAL_K = 4

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
# fit_predict: learn centroids on X_scaled, then return a cluster id per row
df["cluster"] = kmeans.fit_predict(X_scaled)

print(f"Fitted KMeans with k={FINAL_K}")
print("\nUsers per cluster:")
print(df["cluster"].value_counts().sort_index())

## 6. Profile each cluster

We summarize mean behavior **and** churn rate per cluster.
Churn was *not* used to form clusters — comparing churn afterward is a validity check:
if segments are real, risk should differ across them.

In [ ]:
profile_cols = BEHAVIORAL_FEATURES + ["churn"]

# groupby("cluster").mean() = average of each column inside each cluster
cluster_profile = df.groupby("cluster")[profile_cols].mean().round(2)

# Also show size so we know if a cluster is tiny
cluster_profile["n_users"] = df.groupby("cluster").size()
cluster_profile["pct_users"] = (cluster_profile["n_users"] / len(df) * 100).round(1)

cluster_profile

In [ ]:
# Transpose for easier reading: features as rows, clusters as columns
print("Cluster means (features as rows):")
display_cols = BEHAVIORAL_FEATURES + ["churn"]
cluster_profile[display_cols].T

In [ ]:
# Bar chart: churn rate by cluster id (names come in the next section)
churn_by_cluster = df.groupby("cluster")["churn"].mean().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(
    churn_by_cluster.index.astype(str),
    churn_by_cluster.values * 100,
    color=sns.color_palette("muted", n_colors=FINAL_K),
)
ax.set_title("Churn Rate by Cluster (churn not used in clustering)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, max(churn_by_cluster.values * 100) * 1.25)

for bar, rate in zip(bars, churn_by_cluster.values * 100):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{rate:.1f}%",
        ha="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

## 7. Human-readable cluster names

Based on the profile table (with `random_state=42` on this dataset):

| Cluster | Name | Reasoning |
|---|---|---|
| 0 | **Power Users** | Highest sessions, views, likes, shares, notification clicks; shortest `days_since_last_login` (~5 days); ~0% churn |
| 1 | **Casual Browsers** | Mid-low engagement across the board; moderate recency gap (~26 days); modest churn (~7%) |
| 2 | **Steady Engagers** | Strong but not elite activity (sessions/views between Casual and Power); recent logins; ~0% churn |
| 3 | **At-Risk/Dormant** | Lowest engagement, longest time since last login (~39 days); **by far the highest churn (~46%)** |

These names are product language for the same four mathematical groups.

In [ ]:
# Must match src/clustering.py CLUSTER_NAMES (same k + random_state)
CLUSTER_NAMES = {
    0: "Power Users",
    1: "Casual Browsers",
    2: "Steady Engagers",
    3: "At-Risk/Dormant",
}

# map() replaces each integer cluster id with its string name
df["cluster_name"] = df["cluster"].map(CLUSTER_NAMES)

print("Cluster sizes:")
print(df["cluster_name"].value_counts())
print("\nChurn rate by named cluster:")
# sort_values so the riskiest segment appears first
print(
    df.groupby("cluster_name")["churn"]
    .agg(churn_rate="mean", n_users="count")
    .sort_values("churn_rate", ascending=False)
    .assign(churn_rate_pct=lambda x: (x["churn_rate"] * 100).round(1))
)

## 8. Visualize clusters with PCA (2D)

**Important:** PCA here is **only for visualization**. Clustering still runs on all 7 scaled features.
PCA finds 2 axes that capture the most variance so we can scatter-plot in 2D and check whether clusters separate visually.

In [ ]:
# n_components=2 → project 7D scaled features down to 2 numbers per user (for plotting)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    {
        "pc1": X_pca[:, 0],
        "pc2": X_pca[:, 1],
        "cluster_name": df["cluster_name"],
    }
)

# explained_variance_ratio_ = fraction of total variance captured by each PC
var = pca.explained_variance_ratio_
print(f"PC1 explains {var[0]:.1%} of variance")
print(f"PC2 explains {var[1]:.1%} of variance")
print(f"Together: {var.sum():.1%} (rest is compressed away in the 2D plot)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

palette = {
    "Power Users": "#55a868",
    "Steady Engagers": "#4c72b0",
    "Casual Browsers": "#dd8452",
    "At-Risk/Dormant": "#c44e52",
}

# Scatter each cluster with a distinct color
for name, subset in pca_df.groupby("cluster_name"):
    ax.scatter(
        subset["pc1"],
        subset["pc2"],
        s=12,
        alpha=0.45,
        label=name,
        color=palette.get(name, "gray"),
    )

ax.set_title("K-Means Clusters in 2D (PCA projection of 7 behavioral features)")
ax.set_xlabel(f"PC1 ({var[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var[1]:.1%} variance)")
ax.legend(markerscale=2, frameon=True)
plt.tight_layout()
plt.show()

## 9. Save clustered data + models for Day 3

In [ ]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# joblib persists the fitted sklearn objects so Day 3 can reuse them
joblib.dump(scaler, models_dir / "scaler.joblib")
joblib.dump(kmeans, models_dir / "kmeans.joblib")

out_path = PROJECT_ROOT / "data" / "users_with_clusters.csv"
df.to_csv(out_path, index=False)

print(f"Saved scaler  -> {models_dir / 'scaler.joblib'}")
print(f"Saved kmeans  -> {models_dir / 'kmeans.joblib'}")
print(f"Saved dataset -> {out_path}")
print(f"Columns now include: cluster, cluster_name")
df[["user_id", "cluster", "cluster_name", "churn"]].head()

In [ ]:
# Sanity check: reusable helper matches this notebook's labels
from clustering import assign_clusters

check = assign_clusters(pd.read_csv(PROJECT_ROOT / "data" / "users.csv"))
matches = (check["cluster"].values == df["cluster"].values).all()
print(f"assign_clusters() matches notebook labels: {matches}")

## Day 2 summary — so what?

- We segmented **10,000 users into 4 behavioral clusters** using K-Means on 7 scaled engagement features (churn was held out).
- Cluster names: **Power Users**, **Steady Engagers**, **Casual Browsers**, and **At-Risk/Dormant**.
- **Highest churn:** At-Risk/Dormant (~46%) — low activity + long gaps since last login.
- **Lowest churn:** Power Users and Steady Engagers (~0%) — frequent, recent engagement.
- Casual Browsers sit in the middle (~7% churn): not dormant yet, but worth nurturing.

**Takeaway for Day 3:** cluster membership is a strong, leakage-free signal for churn models. Saved artifacts: `data/users_with_clusters.csv`, `models/scaler.joblib`, `models/kmeans.joblib`, plus `src/clustering.py` (`assign_clusters`).